# Beam Search 束搜索

> 自回归解码的经典方法，贪心搜索和 Top-K 采样的增强版。

## 背景
贪心搜索每步只保留 1 个最优候选，容易陷入局部最优。Beam Search 每步保留 K 个最优**序列**，
以序列级对数概率为评分标准。K 越大搜索越广但越慢。

## 算法
1. 初始化：beam = [(score=0, seq=[BOS])]
2. 每步：对 beam 中每条序列生成 top-K 个候选 token，得 B×K 个新序列
3. 取对数概率 top-K 作为新 beam
4. 直到所有 beam 结束或达到 max_length

## 复杂度
- 时间：O(T × K × V × log(K))，T=序列长度，K=beam size，V=词表大小
- 空间：O(K × T)

## 考察点
- 序列级评分 vs token 级评分
- beam size 对多样性的影响（越大越趋近贪心）
- length normalization 防长序列惩罚


In [ ]:
import torch
import torch.nn.functional as F
from dataclasses import dataclass, field
from typing import List, Callable

@dataclass
class Beam:
    score: float
    tokens: list
    finished: bool = False

def beam_search(logits_fn: Callable, beam_size: int = 4, max_len: int = 50,
                eos_token: int = 0, length_penalty: float = 0.0,
                bos_token: int = 1) -> List[Beam]:
    """
    Beam Search 解码。
    logits_fn(tokens: list) -> Tensor[vocab]：给定已生成 token 返回下一个 token 的 logits。
    返回按 score 排序的 beam 列表。
    """
    beams = [Beam(score=0.0, tokens=[bos_token])]

    for step in range(max_len - 1):
        candidates = []
        for beam in beams:
            if beam.finished:
                candidates.append(beam)
                continue
            logits = logits_fn(beam.tokens)
            log_probs = F.log_softmax(logits, dim=-1)
            topk_log_probs, topk_ids = log_probs.topk(beam_size)
            for i in range(beam_size):
                tok = topk_ids[i].item()
                new_score = beam.score + topk_log_probs[i].item()
                new_tokens = beam.tokens + [tok]
                finished = (tok == eos_token)
                candidates.append(Beam(score=new_score, tokens=new_tokens, finished=finished))

        candidates.sort(key=lambda b: b.score / max(len(b.tokens), 1) ** length_penalty, reverse=True)
        beams = candidates[:beam_size]

        if all(b.finished for b in beams):
            break

    return beams


In [ ]:
# ===== 测试验证 =====
torch.manual_seed(42)
vocab_size = 10

def fake_logits_fn(tokens):
    torch.manual_seed(sum(tokens) * 31 + 7)
    return torch.randn(vocab_size)

beams = beam_search(fake_logits_fn, beam_size=3, max_len=5, eos_token=9)
assert 1 <= len(beams) <= 3, f"beam 数量应 1-3, 实际 {len(beams)}"
assert all(len(b.tokens) <= 5 for b in beams), "序列长度不超 max_len"
assert beams[0].score >= beams[-1].score, "应按 score 降序"
print(f"✅ beam_search: {len(beams)} 条 beam, 最优 score={beams[0].score:.4f}")
print(f"  tokens: {beams[0].tokens}")

beams2 = beam_search(fake_logits_fn, beam_size=1, max_len=5, eos_token=9)
assert len(beams2) == 1
print(f"✅ beam_size=1 退化为贪心: {beams2[0].tokens}")

beams3 = beam_search(fake_logits_fn, beam_size=5, max_len=8, eos_token=9, length_penalty=1.0)
assert all(len(b.tokens) <= 8 for b in beams3)
print(f"✅ length_penalty=1.0: {len(beams3)} 条 beam")

beams4 = beam_search(fake_logits_fn, beam_size=4, max_len=3, eos_token=-1)
assert all(not b.finished for b in beams4), "无 EOS 不应 finished"
print(f"✅ 无 EOS: 全部未完成, 长度={len(beams4[0].tokens)}")
print("✅ 全部测试通过")
